# 🎓 Notebook 3: Unsupervised Learning — Customer Behavior Clustering

## AuraCart Retail Analytics — Discovering Hidden Customer Patterns

This notebook applies **K-Means clustering** to discover hidden patterns in customer purchasing behavior. The goal is to group similar transactions/customers based on their numerical characteristics, without using any predefined labels.

### Objectives
1. Apply K-Means Clustering after feature scaling
2. Determine optimal number of clusters using Elbow Method and Silhouette Score
3. Interpret clusters by analyzing centroid characteristics
4. Translate clusters into actionable business insights for AuraCart

### Syllabus Alignment
- Module 5: Unsupervised Learning — Clustering (K-Means, WCSS, Euclidean similarity)

## Step 1: Setup & Load Preprocessed Data

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

import mlflow

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

IMG_DIR = os.path.join('..', 'analysis_images')
ARTIFACTS_DIR = os.path.join('..', 'artifacts')

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load the processed data
df = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'processed_data.csv'))
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (10000, 12)


,category,price,quantity,delivery_status,payment_method,device_type,channel,customer_segment,order_month,order_day_of_week,order_hour,shipping_delay_days
0,Books,45.95,4,Shipped,PayPal,Mobile,Paid Search,VIP,4,5,14,7.0
1,Electronics,403.17,3,Delivered,PayPal,Mobile,Paid Search,Returning,4,5,14,2.0
2,Beauty,317.45,2,Shipped,Credit Card,Mobile,Email,Returning,4,5,14,7.0
3,Home,24.08,3,Shipped,PayPal,Tablet,Social,VIP,4,5,14,4.0
4,Clothing,494.90,1,Delivered,PayPal,Tablet,Organic,VIP,4,5,14,5.0


## Step 2: Prepare Data for Clustering

For K-Means clustering, we select **numerical features** only, as K-Means operates on Euclidean distance. We apply StandardScaler to ensure features with larger ranges don't dominate the distance calculations.

In [3]:
# Select numerical features for clustering
numerical_cols = ['price', 'quantity', 'order_month', 'order_day_of_week', 'order_hour', 'shipping_delay_days']

# Extract the clustering features
X_cluster = df[numerical_cols].copy()
print(f'Clustering features: {numerical_cols}')
print(f'Shape: {X_cluster.shape}')
print(f'\nStatistics before scaling:')
X_cluster.describe()

Clustering features: ['price', 'quantity', 'order_month', 'order_day_of_week', 'order_hour', 'shipping_delay_days']
Shape: (10000, 6)

Statistics before scaling:


,price,quantity,order_month,order_day_of_week,order_hour,shipping_delay_days
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.0,10000.000000
mean,252.550681,2.124700,6.482100,2.977900,14.0,4.007000
std,141.394146,1.254315,3.463478,1.999503,0.0,1.993376
min,5.060000,1.000000,1.000000,0.000000,14.0,1.000000
25%,130.607500,1.000000,3.000000,1.000000,14.0,2.000000
50%,252.910000,2.000000,6.000000,3.000000,14.0,4.000000
75%,374.917500,3.000000,10.000000,5.000000,14.0,6.000000
max,499.930000,9.000000,12.000000,6.000000,14.0,7.000000


In [4]:
# Scale features using StandardScaler
# This is essential because K-Means uses Euclidean distance,
# which is sensitive to feature magnitudes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print('Features scaled with StandardScaler.')
print(f'Scaled data shape: {X_scaled.shape}')
print(f'Mean of scaled features (should be ~0): {X_scaled.mean(axis=0).round(4)}')
print(f'Std of scaled features (should be ~1): {X_scaled.std(axis=0).round(4)}')

Features scaled with StandardScaler.
Scaled data shape: (10000, 6)
Mean of scaled features (should be ~0): [ 0.  0. -0. -0.  0.  0.]
Std of scaled features (should be ~1): [1. 1. 1. 1. 0. 1.]


## Step 3: Selecting the Number of Clusters (k)

The number of clusters cannot be chosen randomly. We use two quantitative methods:

1. **Elbow Method:** Plot the Within-Cluster Sum of Squares (WCSS) against k. The optimal k is at the "elbow" where adding more clusters provides diminishing returns.

2. **Silhouette Score:** Measures both cluster separation and cohesion. A score closer to 1 indicates well-separated, dense clusters.

In [5]:
# Setup MLflow for clustering experiments
mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('AuraCart_Customer_Clustering')

# Test k values from 2 to 10
k_range = range(2, 11)
wcss_values = []
silhouette_values = []

for k in k_range:
    with mlflow.start_run(run_name=f'KMeans_k{k}'):
        kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, 
                        max_iter=300, random_state=42)
        labels = kmeans.fit_predict(X_scaled)
        
        wcss = kmeans.inertia_
        sil_score = silhouette_score(X_scaled, labels)
        
        wcss_values.append(wcss)
        silhouette_values.append(sil_score)
        
        # Log to MLflow
        mlflow.log_param('n_clusters', k)
        mlflow.log_metric('WCSS', wcss)
        mlflow.log_metric('silhouette_score', sil_score)
        
        print(f'k={k}: WCSS={wcss:.2f}, Silhouette Score={sil_score:.4f}')

2026/04/04 13:12:25 INFO mlflow.tracking.fluent: Experiment with name 'AuraCart_Customer_Clustering' does not exist. Creating a new experiment.


k=2: WCSS=42458.22, Silhouette Score=0.1515


k=3: WCSS=37113.39, Silhouette Score=0.1545


k=4: WCSS=33470.55, Silhouette Score=0.1474


k=5: WCSS=30237.77, Silhouette Score=0.1557


k=6: WCSS=27746.30, Silhouette Score=0.1605


k=7: WCSS=26003.43, Silhouette Score=0.1597


k=8: WCSS=24289.53, Silhouette Score=0.1661


k=9: WCSS=22637.25, Silhouette Score=0.1762


k=10: WCSS=21658.68, Silhouette Score=0.1775


In [6]:
# Visualize Elbow Method and Silhouette Scores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow Method
axes[0].plot(k_range, wcss_values, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method for Optimal k', fontsize=14)
axes[0].set_xticks(list(k_range))
axes[0].grid(True, alpha=0.3)

# Silhouette Score
axes[1].plot(k_range, silhouette_values, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score for Optimal k', fontsize=14)
axes[1].set_xticks(list(k_range))
axes[1].grid(True, alpha=0.3)

# Mark the best k based on silhouette score
best_k = list(k_range)[np.argmax(silhouette_values)]
axes[1].axvline(x=best_k, color='green', linestyle='--', linewidth=2, label=f'Best k={best_k}')
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'clustering_optimal_k.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n\u2705 Optimal k based on Silhouette Score: {best_k}')
print(f'   Silhouette Score: {max(silhouette_values):.4f}')


✅ Optimal k based on Silhouette Score: 10
   Silhouette Score: 0.1775


## Step 4: Final Clustering with Optimal k

We apply K-Means with the selected optimal k value and analyze the resulting cluster characteristics.

In [7]:
# Final K-Means with optimal k
# Using k=4 as a reasonable choice based on elbow and business interpretability
# (adjust after running the elbow/silhouette analysis)
optimal_k = best_k

final_kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10,
                       max_iter=300, random_state=42)
cluster_labels = final_kmeans.fit_predict(X_scaled)

# Add cluster labels to the dataframe
df['Cluster'] = cluster_labels

print(f'K-Means clustering completed with k={optimal_k}')
print(f'\nCluster distribution:')
print(df['Cluster'].value_counts().sort_index())

# Final silhouette score
final_sil = silhouette_score(X_scaled, cluster_labels)
print(f'\nFinal Silhouette Score: {final_sil:.4f}')

K-Means clustering completed with k=10

Cluster distribution:
Cluster
0    1054
1    1005
2    1135
3    1123
4     678
5    1086
6    1099
7    1012
8    1008
9     800
Name: count, dtype: int64



Final Silhouette Score: 0.1775


## Step 5: Cluster Interpretation & Centroid Analysis

We examine the centroids (cluster centers) to understand what makes each cluster unique. The centroids represent the "average customer profile" within each cluster.

In [8]:
# Centroid analysis (inverse transform to original scale)
centroids_scaled = final_kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

centroid_df = pd.DataFrame(centroids_original, columns=numerical_cols)
centroid_df.index.name = 'Cluster'

print('\n=== Cluster Centroids (Original Scale) ===')
print(centroid_df.round(2).to_string())


=== Cluster Centroids (Original Scale) ===
          price  quantity  order_month  order_day_of_week  order_hour  shipping_delay_days
Cluster                                                                                   
0        139.69      1.74         3.63               4.62        14.0                 5.33
1        127.86      1.60         9.42               1.70        14.0                 5.49
2        375.64      1.71         3.79               4.40        14.0                 2.57
3        349.46      1.78         9.21               1.26        14.0                 2.25
4        229.35      4.64         6.48               4.61        14.0                 3.38
5        140.65      1.99         3.65               1.35        14.0                 2.49
6        363.36      1.74         9.26               4.48        14.0                 5.60
7        142.29      1.75         9.18               4.61        14.0                 2.26
8        359.45      1.59         3.70        

In [9]:
# Cluster summary statistics
print('\n=== Cluster Summary Statistics ===')
for cluster_id in range(optimal_k):
    cluster_data = df[df['Cluster'] == cluster_id]
    print(f'\n--- Cluster {cluster_id} ({len(cluster_data)} customers, {len(cluster_data)/len(df)*100:.1f}%) ---')
    print(f'  Avg Price: ${cluster_data["price"].mean():.2f}')
    print(f'  Avg Quantity: {cluster_data["quantity"].mean():.2f}')
    print(f'  Avg Shipping Delay: {cluster_data["shipping_delay_days"].mean():.2f} days')
    print(f'  Top Category: {cluster_data["category"].mode().values[0]}')
    print(f'  Top Delivery Status: {cluster_data["delivery_status"].mode().values[0]}')
    print(f'  Top Customer Segment: {cluster_data["customer_segment"].mode().values[0]}')


=== Cluster Summary Statistics ===

--- Cluster 0 (1054 customers, 10.5%) ---
  Avg Price: $139.75
  Avg Quantity: 1.73
  Avg Shipping Delay: 5.33 days
  Top Category: Electronics
  Top Delivery Status: Delivered
  Top Customer Segment: VIP

--- Cluster 1 (1005 customers, 10.1%) ---
  Avg Price: $127.86
  Avg Quantity: 1.60
  Avg Shipping Delay: 5.49 days
  Top Category: Beauty
  Top Delivery Status: Delivered
  Top Customer Segment: VIP

--- Cluster 2 (1135 customers, 11.3%) ---
  Avg Price: $375.64
  Avg Quantity: 1.71
  Avg Shipping Delay: 2.57 days
  Top Category: Electronics
  Top Delivery Status: Delivered
  Top Customer Segment: VIP

--- Cluster 3 (1123 customers, 11.2%) ---
  Avg Price: $349.35
  Avg Quantity: 1.78
  Avg Shipping Delay: 2.25 days
  Top Category: Electronics
  Top Delivery Status: Delivered
  Top Customer Segment: VIP

--- Cluster 4 (678 customers, 6.8%) ---
  Avg Price: $229.35
  Avg Quantity: 4.64
  Avg Shipping Delay: 3.38 days
  Top Category: Beauty
  Top D

In [10]:
# Visualize cluster characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Price vs Quantity by Cluster
for c in range(optimal_k):
    mask = cluster_labels == c
    axes[0, 0].scatter(df.loc[mask, 'price'], df.loc[mask, 'quantity'],
                       alpha=0.3, s=10, label=f'Cluster {c}')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Quantity')
axes[0, 0].set_title('Price vs Quantity by Cluster')
axes[0, 0].legend()

# Price distribution by cluster
for c in range(optimal_k):
    mask = cluster_labels == c
    axes[0, 1].hist(df.loc[mask, 'price'], bins=30, alpha=0.5, label=f'Cluster {c}')
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Price Distribution by Cluster')
axes[0, 1].legend()

# Shipping delay by cluster
sns.boxplot(x='Cluster', y='shipping_delay_days', data=df, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Shipping Delay by Cluster')

# Centroid radar-style comparison (bar chart)
centroid_df.plot(kind='bar', ax=axes[1, 1], width=0.8)
axes[1, 1].set_title('Centroid Feature Values by Cluster')
axes[1, 1].set_xlabel('Cluster')
axes[1, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'cluster_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

In [11]:
# Cross-tabulation: Clusters vs Customer Segment & Delivery Status
print('\n=== Cluster vs Customer Segment ===')
ct_segment = pd.crosstab(df['Cluster'], df['customer_segment'], normalize='index') * 100
print(ct_segment.round(1).to_string())

print('\n=== Cluster vs Delivery Status ===')
ct_delivery = pd.crosstab(df['Cluster'], df['delivery_status'], normalize='index') * 100
print(ct_delivery.round(1).to_string())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ct_segment.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2')
axes[0].set_title('Customer Segment Distribution per Cluster (%)')
axes[0].set_ylabel('Percentage (%)')
axes[0].legend(title='Segment')

ct_delivery.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set3')
axes[1].set_title('Delivery Status Distribution per Cluster (%)')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Delivery Status')

plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'cluster_crosstab.png'), dpi=150, bbox_inches='tight')
plt.show()


=== Cluster vs Customer Segment ===
customer_segment  New  Returning   VIP
Cluster                               
0                 6.1       41.1  52.8
1                 5.1       44.3  50.6
2                 6.1       45.6  48.4
3                 4.7       43.3  52.0
4                 5.9       41.9  52.2
5                 5.7       42.2  52.1
6                 5.9       41.9  52.1
7                 7.1       40.6  52.3
8                 5.1       43.2  51.8
9                 4.9       44.4  50.7

=== Cluster vs Delivery Status ===
delivery_status  Delivered  Pending  Returned  Shipped
Cluster                                               
0                     70.1      6.4       4.7     18.8
1                     71.2      4.6       5.2     19.0
2                     69.6      4.8       4.1     21.5
3                     70.3      5.1       6.5     18.1
4                     71.8      3.5       5.3     19.3
5                     69.2      6.0       4.1     20.7
6                  

## Step 6: Business Insights & AuraCart Recommendations

Based on the cluster analysis, we translate the discovered customer segments into actionable business strategies for AuraCart.

### Cluster Profiles & Recommended Strategies

| Cluster | Type | Strategy |
|---------|------|----------|
| **0** | Examine centroid values after running | Targeted marketing based on behavior |
| **1** | Examine centroid values after running | Personalized promotions |
| **2** | Examine centroid values after running | Dynamic pricing strategies |
| **3** | Examine centroid values after running | Inventory planning & recommendations |

**Note:** Update this table after running the notebook with actual cluster characteristics.

### Potential Business Applications:
1. **Targeted Marketing Campaigns:** Tailor email content and ads based on cluster behavior
2. **Personalized Promotions:** Offer discounts on preferred categories for each cluster
3. **Dynamic Pricing Strategies:** Adjust pricing based on price sensitivity of each cluster
4. **Inventory Planning:** Stock products based on cluster purchasing patterns
5. **Churn Mitigation:** Monitor clusters with high return rates for early intervention

In [12]:
# Save clustering artifacts
joblib.dump(final_kmeans, os.path.join(ARTIFACTS_DIR, 'kmeans_model.pkl'))
joblib.dump(scaler, os.path.join(ARTIFACTS_DIR, 'clustering_scaler.pkl'))

print('\u2705 Clustering artifacts saved:')
print('  - kmeans_model.pkl')
print('  - clustering_scaler.pkl')

✅ Clustering artifacts saved:
  - kmeans_model.pkl
  - clustering_scaler.pkl


## Summary

In this notebook, we have:

1. **Prepared numerical features** for clustering with StandardScaler
2. **Determined optimal k** using both the Elbow Method (WCSS) and Silhouette Score
3. **Applied K-Means clustering** to segment AuraCart customers
4. **Analyzed cluster centroids** to identify distinct customer profiles
5. **Cross-tabulated clusters** with customer segments and delivery statuses
6. **Provided business insights** for targeted marketing, personalized promotions, dynamic pricing, and inventory planning
7. **Tracked all experiments** with MLflow for reproducibility